# Camera Test — Shirt Colour Detection

Live webcam demo that detects a person and displays their predicted shirt colour.

In [ ]:
import sys, os, importlib

# Add project root to path so utils/ is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import utils.test_utils
importlib.reload(utils.test_utils)
from utils.test_utils import ShirtColorDetector

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
MODEL_PATH = os.path.join(MODELS_DIR, "student_int8.pth")

print(f"Models dir: {MODELS_DIR}")
print(f"Using checkpoint: {MODEL_PATH}")

Models dir: c:\Users\lifei\OneDrive\Desktop\Personal Projects\BBAIRL\Clothing-Color-Recognition-ML-With-Synthetic-Dataset\models
Using checkpoint: c:\Users\lifei\OneDrive\Desktop\Personal Projects\BBAIRL\Clothing-Color-Recognition-ML-With-Synthetic-Dataset\models\student_int8.pth


: 

## Live Webcam Feed

Close the window to stop the camera.

In [ ]:
import cv2

CAMERA_INDEX = 1
WINDOW_NAME  = "Shirt Colour Detection"
MAX_DIM      = 720

with ShirtColorDetector(models_dir=MODELS_DIR) as detector:
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open camera {CAMERA_INDEX}")

    # Minimize internal buffer so we always get the latest frame
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    # Read one frame to get the native resolution, then resize window
    ret, frame = cap.read()
    if ret:
        h, w = frame.shape[:2]
        scale = min(MAX_DIM / w, MAX_DIM / h)
        cv2.resizeWindow(WINDOW_NAME, int(w * scale), int(h * scale))

    print("Camera opened. Close the window to quit.")
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            annotated, results = detector.process_frame(frame)
            cv2.imshow(WINDOW_NAME, annotated)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            if cv2.getWindowProperty(WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1:
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Camera released.")

## Video File Test

Set `VIDEO_PATH` to your video file and run the cell below.

In [3]:
import cv2
import time

VIDEO_PATH  = os.path.join(PROJECT_ROOT, "videos", "colors_video_hd.mp4")
WINDOW_NAME = "Test Shirt Colour Detection"
MAX_DIM     = 720

with ShirtColorDetector(models_dir=MODELS_DIR) as detector:
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

    fps          = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = min(MAX_DIM / w, MAX_DIM / h)
    cv2.resizeWindow(WINDOW_NAME, int(w * scale), int(h * scale))

    print(f"Video opened (fps={fps:.1f}, frames={total_frames}). Close the window to stop early.")

    start_time  = time.perf_counter()
    frame_idx   = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            # Skip frames that are behind real-time playback
            elapsed      = time.perf_counter() - start_time
            target_frame = int(elapsed * fps)
            if frame_idx < target_frame:
                continue

            annotated, results = detector.process_frame(frame)
            cv2.imshow(WINDOW_NAME, annotated)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            if cv2.getWindowProperty(WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1:
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Done.")

Video opened (fps=30.0, frames=4625). Close the window to stop early.
Done.
